# 09b — Study Area Overview (world map + per-city OSM panels)

One figure, two rows, following the `paper-figure-style` house
convention (final-size canvas, trimmed spines, direct labelling, panel
letters):

- **Row 1** -- a single world map with all four study-area cities
  pinpointed, each encoded by both colour and marker shape (redundant
  coding, so it survives greyscale/CVD), with a legend naming them.
- **Row 2** -- one panel per city, side by side (`City A | City B |
  City C | City D`), each on a real OpenStreetMap basemap (Cartopy's
  `OSM` tile source), showing that city's actual study-area boundary
  and its positive (crash) / negative (generated) points, with
  longitude/latitude tick labels on both axes.

**Read-only.** Reads only `paths.yaml`'s per-city `boundary_geojson`,
`positive_points_csv`, and `negative_points_csv` -- nothing under
`src/` or `configs/` is written to. Output goes to
`OUTPUTS_DIR/paper_figures/` (the same directory notebook `09` uses),
so it lands alongside the other paper figures. Row 2's OSM tiles are
fetched live (needs internet access, e.g. on Colab) and cached by
Cartopy for the session.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running on Colab -- skipping drive mount (paths.yaml must already resolve locally).")

In [ ]:
!pip install -q pandas numpy matplotlib seaborn pyyaml geopandas shapely cartopy

## 1. Paths and output directory

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
FIG_DIR = OUTPUTS_DIR / "paper_figures"   # same dir notebook 09 uses
FIG_DIR.mkdir(parents=True, exist_ok=True)

CITY_DISPLAY_NAME = {
    "bogor": "Bogor, Indonesia",
    "warsaw": "Warsaw, Poland",
    "krakow": "Krak\u00f3w, Poland",
    "somerville": "Somerville, USA",
}

print("Cities:", CITIES)
print("FIG_DIR:", FIG_DIR)

## 2. Load per-city boundary + positive/negative points

Each city's `boundary_geojson` gives the actual study-area polygon (used both to compute the city's map-marker position -- its centroid -- and to draw the inset outline). Points are loaded straight from `positive_points_csv`/`negative_points_csv`, no filtering beyond what those files already contain.

In [ ]:
import geopandas as gpd
import pandas as pd

city_data = {}
for city in CITIES:
    pc = paths_cfg["per_city"][city]
    entry = {}

    boundary_path = Path(pc["boundary_geojson"])
    if boundary_path.exists():
        gdf = gpd.read_file(boundary_path)
        entry["boundary"] = gdf.geometry.unary_union
        c = entry["boundary"].centroid
        entry["center"] = (c.x, c.y)  # (lon, lat)
    else:
        print(f"  [skip] {boundary_path} not found for {city}.")
        entry["boundary"] = None
        entry["center"] = None

    pos_path, neg_path = Path(pc["positive_points_csv"]), Path(pc["negative_points_csv"])
    entry["positive"] = pd.read_csv(pos_path) if pos_path.exists() else None
    entry["negative"] = pd.read_csv(neg_path) if neg_path.exists() else None
    if entry["positive"] is None:
        print(f"  [skip] {pos_path} not found for {city}.")
    if entry["negative"] is None:
        print(f"  [skip] {neg_path} not found for {city}.")

    city_data[city] = entry
    n_pos = len(entry["positive"]) if entry["positive"] is not None else 0
    n_neg = len(entry["negative"]) if entry["negative"] is not None else 0
    center = entry["center"]
    print(f"{CITY_DISPLAY_NAME.get(city, city)}: center={center}, "
          f"n_positive={n_pos}, n_negative={n_neg}")

## 3. House figure style (inline, self-contained, same as notebook 09)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns


class _PaperStyle:
    FULL_W = 7.0
    FS_TICK, FS_LABEL, FS_TITLE, FS_LETTER, FS_LEGEND = 6.5, 7.5, 8.0, 9.5, 6.5
    MUTED = ["#9fd4c0", "#c3b49a", "#8a7358", "#9aa4cd", "#4a4a73",
             "#8ecae0", "#f2a58c", "#3f8f7d"]
    FOCAL = "#8c2f2f"
    LW_SPINE = 0.7

    def apply(self, font="Liberation Sans"):
        sns.set_theme(style="ticks")
        mpl.rcParams.update({
            "font.family": "sans-serif",
            "font.sans-serif": [font, "Arial", "Helvetica", "DejaVu Sans"],
            "font.size": self.FS_TICK,
            "axes.labelsize": self.FS_LABEL,
            "axes.titlesize": self.FS_TITLE,
            "xtick.labelsize": self.FS_TICK,
            "ytick.labelsize": self.FS_TICK,
            "legend.fontsize": self.FS_LEGEND,
            "axes.linewidth": self.LW_SPINE,
            "axes.grid": False,
            "axes.facecolor": "white",
            "figure.facecolor": "white",
            "legend.frameon": False,
            "savefig.dpi": 300,
            "savefig.bbox": "tight",
            "savefig.pad_inches": 0.02,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        })

    def finish(self, ax, trim=True):
        """Despine + trim for a PLAIN (non-geographic) axes only -- never
        call this on a Cartopy GeoAxes, which manages its own outline/spine
        system and isn't compatible with seaborn's despine."""
        sns.despine(ax=ax, top=True, right=True, trim=trim)

    def panel_letter(self, fig, ax, letter, dx=-0.06, dy=1.08):
        ax.text(dx, dy, letter, transform=ax.transAxes,
                fontsize=self.FS_LETTER, fontweight="bold", ha="left", va="bottom")

    def sparse_yticks(self, ax, n=4):
        ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=n, prune=None))

    def save(self, fig, name):
        fig.savefig(FIG_DIR / f"{name}.pdf")
        fig.savefig(FIG_DIR / f"{name}.png")
        print(f"  [saved] {name}.pdf + {name}.png")


ps = _PaperStyle()
ps.apply()

## 4. World map + a 2x2 grid of per-city OSM panels

```
World map (a)
Map 1 (b) | Map 2 (c)
Map 3 (d) | Map 4 (e)
```

Row 1 is one world map with all four cities pinpointed, each city
coded by both **colour and marker shape** (so identity survives
greyscale printing or colour-vision deficiency, not colour alone) --
left at Cartopy's true (2:1) aspect and **never stretched**, so
continents are never distorted, even though that leaves it narrower
than the grid (letterboxed) rather than edge-to-edge. Below it, the
four cities sit in a 2x2 grid, each on a real OpenStreetMap basemap,
with the study-area boundary and positive/negative points.

**Each city panel is "zoomed to layer,"** exactly to that city's own
positive/negative point cloud (not the official study-area boundary,
which can be larger or off-center relative to where points actually
landed), padded just enough that no point sits on the frame edge.
**Every panel is a true square, and all four share the same box
aspect** -- the padded extent uses the LARGER of the point cloud's own
ground-distance lon/lat spans for both axes, with the usual
ground-distance correction (a degree of longitude covers less ground
than a degree of latitude away from the equator) applied per axis so
the square is true-to-scale, not a degree-square. Each city keeps its
own zoom/scale (there is no shared scale across cities), shown
explicitly as a **scale bar**, and the OSM zoom level is chosen per
city to match how tightly it's framed.

In [ ]:
import math
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.img_tiles as cimgt
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.lines import Line2D
from matplotlib.patheffects import withStroke

POS_COLOR, NEG_COLOR = ps.MUTED[4], ps.FOCAL  # same pairing as F1
CITY_COLOR = {c: ps.MUTED[i % len(ps.MUTED)] for i, c in enumerate(CITIES)}
CITY_MARKER = {c: m for c, m in zip(CITIES, ["o", "s", "^", "D", "P", "X"])}

KM_PER_DEG_LAT = 111.32  # ~constant; longitude's km/degree shrinks by cos(latitude)


def _zoom_for_span_km(span_km):
    """OSM zoom level roughly matched to how tight this city's own
    point-cloud framing is -- a city with a 1 km spread gets zoomed in
    much further than one spanning 15 km, rather than sharing one zoom."""
    for limit, zoom in [(0.3, 17), (0.6, 16), (1.2, 15), (2.5, 14),
                       (5, 13), (10, 12), (20, 11), (40, 10)]:
        if span_km <= limit:
            return zoom
    return 9


def _point_bounds(pos_df, neg_df):
    lons, lats = [], []
    for df in (pos_df, neg_df):
        if df is not None and {"lon", "lat"}.issubset(df.columns):
            lons.extend(df["lon"].tolist())
            lats.extend(df["lat"].tolist())
    if not lons:
        return None
    return min(lons), min(lats), max(lons), max(lats)


def _add_scale_bar(ax, extent, mean_lat, km_per_deg_lon):
    """A simple ground-truth scale bar, drawn in data (lon/lat)
    coordinates -- Cartopy has no built-in one. Picks a round bar length
    (~1/4 of the panel's own width) and draws it bottom-left, in the
    same PlateCarree transform the map data uses, so it reads correctly
    however this city happens to be zoomed."""
    width_km = (extent[1] - extent[0]) * km_per_deg_lon
    candidates_km = [0.01, 0.02, 0.05, 0.1, 0.2, 0.25, 0.5, 1, 2, 5, 10, 20, 50]
    bar_km = min(candidates_km, key=lambda c: abs(c - width_km / 4))
    bar_deg = bar_km / km_per_deg_lon

    lon_span, lat_span = extent[1] - extent[0], extent[3] - extent[2]
    x0 = extent[0] + lon_span * 0.06
    y0 = extent[2] + lat_span * 0.07
    halo = [withStroke(linewidth=2.5, foreground="white")]

    ax.plot([x0, x0 + bar_deg], [y0, y0], color="#111111", lw=2.2,
            solid_capstyle="butt", transform=ccrs.PlateCarree(), zorder=6,
            path_effects=halo)
    cap_h = lat_span * 0.014
    for x in (x0, x0 + bar_deg):
        ax.plot([x, x], [y0 - cap_h, y0 + cap_h], color="#111111", lw=1.6,
                transform=ccrs.PlateCarree(), zorder=6, path_effects=halo)
    label = f"{bar_km:g} km" if bar_km >= 1 else f"{int(round(bar_km * 1000))} m"
    ax.text(x0 + bar_deg / 2, y0 + lat_span * 0.022, label, transform=ccrs.PlateCarree(),
           ha="center", va="bottom", fontsize=ps.FS_TICK, fontweight="bold",
           color="#111111", zorder=6, path_effects=halo)


def plot_city_panel(ax, city, entry, letter):
    boundary, pos_df, neg_df = entry["boundary"], entry["positive"], entry["negative"]
    geo = entry["_geo"]
    extent, mean_lat, km_per_deg_lon = geo["extent"], geo["mean_lat"], geo["km_per_deg_lon"]

    ax.set_extent(extent, crs=ccrs.PlateCarree())
    # Every panel is a true square (see compute_city_extent), so this cell
    # should already be filled exactly -- "C" is just a safety net against
    # any hairline leftover from floating-point rounding, split evenly
    # rather than pinned to one edge.
    ax.set_anchor("C")
    ax.add_image(cimgt.OSM(), _zoom_for_span_km(max(geo["lon_span_km"], geo["lat_span_km"])),
                interpolation="bilinear")
    # add_image() only QUEUES the tile fetch as an "image factory" -- the
    # actual tile download + imshow() call is deferred until this axes is
    # first drawn. That deferred imshow() silently calls set_aspect('equal')
    # internally (matplotlib's default imshow behaviour), which would
    # clobber any aspect we set here *no matter where in this function we
    # set it*, because the real reset happens later, at first draw, not at
    # this add_image() call site. Forcing that first draw NOW consumes the
    # image factory (cartopy marks it done and never re-triggers imshow on
    # later draws), so our own set_aspect() below -- applied strictly AFTER
    # this forced draw -- is the one that actually sticks.
    ax.figure.canvas.draw()
    # Cartopy's own aspect-lock for a PlateCarree axes uses the RAW DEGREE
    # ratio (lon_range_deg / lat_range_deg), not a ground-distance-corrected
    # one -- so without this line, Cartopy re-shrinks the axes box to its
    # own (uncorrected) notion of "equal aspect", ignoring the ground-true
    # `extent`/`aspect` this module carefully computed above. This is the
    # standard regional-PlateCarree correction: it tells Cartopy the box
    # aspect that actually corresponds to true distances at this latitude,
    # so the box we hand it (sized to match `geo["aspect"]`) renders at its
    # full allocated size instead of being letterboxed.
    ax.set_aspect(1.0 / math.cos(math.radians(mean_lat)))

    parts = [boundary] if boundary.geom_type == "Polygon" else list(boundary.geoms)
    for part in parts:
        bx, by = part.exterior.xy
        ax.plot(bx, by, color="#1a1a1a", lw=1.2, transform=ccrs.PlateCarree(), zorder=3)

    if neg_df is not None and {"lon", "lat"}.issubset(neg_df.columns):
        ax.scatter(neg_df["lon"], neg_df["lat"], s=6, color=NEG_COLOR, alpha=0.85,
                   edgecolor="none", transform=ccrs.PlateCarree(), zorder=4, label="negative")
    if pos_df is not None and {"lon", "lat"}.issubset(pos_df.columns):
        ax.scatter(pos_df["lon"], pos_df["lat"], s=6, color=POS_COLOR, alpha=0.85,
                   edgecolor="none", transform=ccrs.PlateCarree(), zorder=5, label="positive")

    ax.set_xticks(np.linspace(extent[0], extent[1], 3), crs=ccrs.PlateCarree())
    ax.set_yticks(np.linspace(extent[2], extent[3], 3), crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter(number_format=".3f"))
    ax.yaxis.set_major_formatter(LatitudeFormatter(number_format=".3f"))
    ax.tick_params(labelsize=ps.FS_TICK, length=2.5)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333"); spine.set_linewidth(0.8)
    _add_scale_bar(ax, extent, mean_lat, km_per_deg_lon)

    ax.set_title(CITY_DISPLAY_NAME.get(city, city), fontsize=ps.FS_LABEL, fontweight="bold")
    if letter:
        ps.panel_letter(fig, ax, letter)


def compute_city_extent(entry):
    """Square, not rectangle: every city panel gets the SAME box aspect
    (1:1), so the 2x2 grid is uniform and no row needs sizing to its
    tallest-relative city. The square is true-to-scale ground distance
    (not a degree-square) -- span_km is the LARGER of the point cloud's own
    lon/lat ground spans (so nothing gets cropped), padded, then applied
    equally to both axes via each axis's own km-per-degree factor.
    Computed once, up front, for every city -- both so plot_city_panel
    doesn't redo it AND so the figure/grid can use one shared cell size
    before any axes are created (see below)."""
    boundary, pos_df, neg_df = entry["boundary"], entry["positive"], entry["negative"]
    bounds = _point_bounds(pos_df, neg_df)
    if bounds is None:
        bounds = boundary.bounds  # fall back to the boundary polygon only if no points exist
    minx, miny, maxx, maxy = bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    mean_lat = cy
    km_per_deg_lon = KM_PER_DEG_LAT * math.cos(math.radians(mean_lat))

    lon_range_km = (maxx - minx) * km_per_deg_lon
    lat_range_km = (maxy - miny) * KM_PER_DEG_LAT
    pad = 1.20
    span_km = max(lon_range_km, lat_range_km, 0.02) * pad
    half_lon = (span_km / 2) / km_per_deg_lon
    half_lat = (span_km / 2) / KM_PER_DEG_LAT
    extent = [cx - half_lon, cx + half_lon, cy - half_lat, cy + half_lat]
    return {"extent": extent, "mean_lat": mean_lat, "km_per_deg_lon": km_per_deg_lon,
            "lon_span_km": span_km, "lat_span_km": span_km, "aspect": 1.0}


N_CITY_COLS = 2  # 2x2 grid of city panels below the world map
n_city_rows = -(-len(CITIES) // N_CITY_COLS)  # ceil

# Precompute every city's real extent BEFORE building the figure, so the
# grid can be laid out from real numbers instead of a guessed constant --
# that guess is exactly what produced the huge empty gaps in an earlier
# version (city panels letterboxed inside uniform-but-wrong-sized cells).
for city in CITIES:
    entry = city_data[city]
    entry["_geo"] = compute_city_extent(entry) if entry["boundary"] is not None else None

FIG_WIDTH_IN = ps.FULL_W * 1.3
PAD_IN = 0.14    # deliberate small gap between panels -- everything else is exact, not guessed
LEGEND_H_IN = 0.35
col_w_in = (FIG_WIDTH_IN - PAD_IN * (N_CITY_COLS - 1)) / N_CITY_COLS

# GridSpec's height_ratios + hspace, even combined with layout="constrained",
# turned out NOT to reliably reproduce each Cartopy GeoAxes' true rendered
# size once aspect-locking is involved -- every attempt left either
# unexplained gaps or clipped panels. Placing every axes manually via
# fig.add_axes([left, bottom, width, height]) in exact, self-computed
# figure-fraction coordinates removes that ambiguity entirely. Every city
# panel is a true SQUARE (compute_city_extent always returns aspect=1.0),
# so every row shares the same height -- no more per-row "tallest city"
# sizing or leftover-slack anchoring needed.
world_h_in = FIG_WIDTH_IN / 2.0  # true 2:1 aspect at full figure width, undistorted
row_h_in = [col_w_in] * n_city_rows  # square cells: row height == column width

FIG_HEIGHT_IN = world_h_in + PAD_IN + sum(row_h_in) + PAD_IN * n_city_rows + LEGEND_H_IN
fig = plt.figure(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))


def _rect(x_in, y_from_top_in, w_in, h_in):
    """(x from left, y from TOP), both in inches -> [left, bottom, width, height]
    in figure-fraction (matplotlib's add_axes coordinate system, y from bottom)."""
    return [x_in / FIG_WIDTH_IN, (FIG_HEIGHT_IN - y_from_top_in - h_in) / FIG_HEIGHT_IN,
            w_in / FIG_WIDTH_IN, h_in / FIG_HEIGHT_IN]


cursor_y_in = 0.0

# --- World map, full width, at its own true (2:1) aspect -- NOT stretched ---
ax_world = fig.add_axes(_rect(0, cursor_y_in, FIG_WIDTH_IN, world_h_in), projection=ccrs.PlateCarree())
ax_world.set_global()
ax_world.add_feature(cfeature.LAND, facecolor="#eee8dd", zorder=0)
ax_world.add_feature(cfeature.OCEAN, facecolor="#dbe7ef", zorder=0)
ax_world.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="0.4", zorder=1)
ax_world.add_feature(cfeature.BORDERS, linewidth=0.25, edgecolor="0.6", zorder=1)
ax_world.set_xticks(np.arange(-180, 181, 60), crs=ccrs.PlateCarree())
ax_world.set_yticks(np.arange(-90, 91, 30), crs=ccrs.PlateCarree())
ax_world.xaxis.set_major_formatter(LongitudeFormatter())
ax_world.yaxis.set_major_formatter(LatitudeFormatter())
ax_world.tick_params(labelsize=ps.FS_TICK, length=2.5)
ax_world.set_title("Study area cities", fontsize=ps.FS_TITLE, fontweight="bold")
ps.panel_letter(fig, ax_world, "a")

world_legend = []
for city in CITIES:
    entry = city_data[city]
    if entry["center"] is None:
        print(f"  [skip] {city} -- no boundary/center available for map placement.")
        continue
    lon, lat = entry["center"]
    ax_world.scatter([lon], [lat], s=45, color=CITY_COLOR[city], marker=CITY_MARKER[city],
                     edgecolor="white", linewidth=0.9, zorder=5, transform=ccrs.PlateCarree())
    world_legend.append(Line2D([], [], marker=CITY_MARKER[city], color="none",
                               markerfacecolor=CITY_COLOR[city], markeredgecolor="white",
                               markersize=7, label=CITY_DISPLAY_NAME.get(city, city)))
ax_world.legend(handles=world_legend, loc="lower left", fontsize=ps.FS_LEGEND,
               frameon=False, handletextpad=0.6, ncol=2)
cursor_y_in += world_h_in + PAD_IN

# --- City rows, 2 per row (Map1|Map2 / Map3|Map4 / ...) ---
panel_letters = "bcdefg"
for r in range(n_city_rows):
    row_cities = CITIES[r * N_CITY_COLS:(r + 1) * N_CITY_COLS]
    cursor_x_in = 0.0
    for ci, city in enumerate(row_cities):
        idx = r * N_CITY_COLS + ci
        entry = city_data[city]
        if entry["_geo"] is not None:
            rect = _rect(cursor_x_in, cursor_y_in, col_w_in, row_h_in[r])
            ax_city = fig.add_axes(rect, projection=ccrs.PlateCarree())
            plot_city_panel(ax_city, city, entry, panel_letters[idx] if idx < len(panel_letters) else "")
        else:
            print(f"  [skip] {city} -- no boundary polygon, panel skipped.")
        cursor_x_in += col_w_in + PAD_IN
    cursor_y_in += row_h_in[r] + PAD_IN

# --- Legend row ---
ax_legend = fig.add_axes(_rect(0, cursor_y_in, FIG_WIDTH_IN, LEGEND_H_IN))
ax_legend.axis("off")
point_legend = [
    Line2D([], [], marker="o", color="none", markerfacecolor=POS_COLOR, markersize=6, label="positive (crash)"),
    Line2D([], [], marker="o", color="none", markerfacecolor=NEG_COLOR, markersize=6, label="negative (generated)"),
]
ax_legend.legend(handles=point_legend, loc="center", ncol=2, fontsize=ps.FS_LEGEND, frameon=False)

ps.save(fig, "F0_study_area_overview")
plt.show()

In [ ]:
print("Figures in", FIG_DIR, ":")
for p in sorted(FIG_DIR.glob("F0_study_area_overview*")):
    print(" -", p.name)